# Text Cleaning Pipeline

## Preparing digitised texts for AI search at the British Library

The British Library has digitised thousands of texts — manuscripts, research papers, exhibition guides, historical records. But digital does not mean clean.

OCR software introduces artefacts. Old HTML formatting lingers. Invisible characters hide in plain sight. Before any AI can read these texts, a data engineer must clean them.

This is not glamorous work. It is, however, essential. A retrieval system built on dirty text will return dirty results. An LLM fed invisible garbage characters will burn through your token budget without producing anything useful. The cleaning pipeline is where data engineering meets real money.

## Loading the raw data

We have a CSV of roughly 2,000 digitised texts from the British Library's collection. Each row contains a text excerpt along with metadata: source type, title, author, and year of publication.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/bl_digitised_texts_raw.csv")
print(f"Loaded {len(df)} texts")
df.head()

In [ ]:
df["source_type"].value_counts()

Let us look at a few raw texts to see what we are dealing with.

In [ ]:
# Pick a few texts to inspect
for i in [0, 10, 50, 100, 200]:
    print(f"--- Text {df.iloc[i]['text_id']} ({df.iloc[i]['source_type']}) ---")
    print(df.iloc[i]["raw_text"][:300])
    print()

Already you can spot problems: HTML tags, page headers, and text that looks clean but may be hiding something. Let us investigate systematically.

## Step 1: HTML tag removal

Many of these texts were originally stored as HTML. The digitisation process stripped most of it, but tags like `<p>`, `<em>`, and `<br/>` have survived. We need to remove them.

Two tools for this:
1. `html.unescape()` converts HTML entities (`&amp;`, `&lt;`, `&nbsp;`) back to their plain text equivalents
2. A regular expression strips any remaining HTML tags

In [ ]:
import html
import re

# Example text with HTML artefacts
sample = "<p>The feudal system in England&nbsp;following the <em>Norman Conquest</em> of 1066<br/>fundamentally restructured land ownership.</p>"
print("Before:", sample)

In [ ]:
# Step 1a: Unescape HTML entities
step1 = html.unescape(sample)
print("After unescape:", step1)

In [ ]:
# Step 1b: Remove HTML tags with regex
step2 = re.sub(r"<[^>]+>", "", step1)
print("After tag removal:", step2)

The regex `<[^>]+>` matches any opening angle bracket, followed by one or more characters that are not a closing angle bracket, followed by a closing angle bracket. In plain English: anything that looks like an HTML tag.

Let us count how many texts contain HTML tags.

In [ ]:
html_pattern = re.compile(r"<[^>]+>")
has_html = df["raw_text"].apply(lambda t: bool(html_pattern.search(t)))
print(f"Texts with HTML tags: {has_html.sum()} out of {len(df)} ({has_html.mean():.1%})")

## Step 2: Unicode normalisation

Unicode is the standard that maps every character in every writing system to a unique number. The problem is that some characters can be represented in more than one way. The letter "e" with an accent can be stored as a single character or as the letter "e" followed by a combining accent mark. They look identical on screen but are different sequences of bytes.

This matters for search and deduplication: if two texts use different representations of the same character, a naive comparison will treat them as different.

Unicode normalisation converts all representations to a canonical form. We use NFKC (Normalisation Form Compatibility Composition), which is the most aggressive: it replaces compatibility characters with their standard equivalents and composes characters where possible.

In [ ]:
import unicodedata

# Two representations of the same word
word_a = "caf\u00e9"     # single character: e-with-accent
word_b = "cafe\u0301"    # two characters: e + combining accent

print(f"word_a: {word_a!r}  (length {len(word_a)})")
print(f"word_b: {word_b!r}  (length {len(word_b)})")
print(f"Equal? {word_a == word_b}")

In [ ]:
# After normalisation, they become identical
norm_a = unicodedata.normalize("NFKC", word_a)
norm_b = unicodedata.normalize("NFKC", word_b)

print(f"norm_a: {norm_a!r}  (length {len(norm_a)})")
print(f"norm_b: {norm_b!r}  (length {len(norm_b)})")
print(f"Equal? {norm_a == norm_b}")

## Step 3: The invisible character problem

This is the part that catches people out. A text can look perfectly clean to the naked eye and still be riddled with invisible characters.

OCR software and web scrapers commonly introduce:
- `\xa0` — the non-breaking space, which looks like a regular space but is a different byte
- `\u200b` — the zero-width space, which is literally invisible and takes up no horizontal space
- `\u200c` and `\u200d` — zero-width non-joiner and joiner, relics of complex script rendering

These characters inflate token counts. When you send text to an LLM API, you pay per token. Invisible characters still count as tokens. They are, in the most literal sense, a waste of money.

The tool that reveals them is `repr()`.

In [ ]:
# This text looks clean...
sneaky = "The British\xa0Library holds over\u200b 170 million\xa0items in its\u200b collection."
print("Looks fine:", sneaky)

In [ ]:
# But repr() tells the truth
print("repr() reveals:", repr(sneaky))

There they are: `\xa0` (non-breaking space) and `\u200b` (zero-width space), hiding in what appeared to be perfectly normal text.

Let us measure the impact. A rough token estimate is to split on whitespace — not perfect, but good enough to illustrate the point.

In [ ]:
# Rough token count: split on whitespace
tokens_before = len(sneaky.split())

# Clean it
cleaned = sneaky.replace("\xa0", " ").replace("\u200b", "")
tokens_after = len(cleaned.split())

print(f"Characters before: {len(sneaky)}, after: {len(cleaned)}")
print(f"Tokens before: {tokens_before}, after: {tokens_after}")
print(f"repr() after cleaning: {repr(cleaned)}")

Let us check how widespread this problem is in the actual dataset.

In [ ]:
invisible_chars = ["\xa0", "\u200b", "\u200c", "\u200d", "\ufeff"]

for char in invisible_chars:
    count = df["raw_text"].apply(lambda t: t.count(char)).sum()
    affected = df["raw_text"].apply(lambda t: char in t).sum()
    print(f"{repr(char):8s}: {count:5d} occurrences in {affected:4d} texts")

Let us see the token impact across the whole dataset.

In [ ]:
def count_rough_tokens(text):
    """Rough token count by splitting on whitespace."""
    return len(text.split())

def strip_invisible(text):
    """Remove common invisible characters."""
    for char in invisible_chars:
        text = text.replace(char, " " if char == "\xa0" else "")
    # Collapse multiple spaces
    text = re.sub(r" {2,}", " ", text)
    return text

tokens_before = df["raw_text"].apply(count_rough_tokens).sum()
tokens_after = df["raw_text"].apply(lambda t: count_rough_tokens(strip_invisible(t))).sum()

print(f"Total tokens before: {tokens_before:,}")
print(f"Total tokens after:  {tokens_after:,}")
print(f"Tokens saved:        {tokens_before - tokens_after:,} ({(tokens_before - tokens_after) / tokens_before:.1%})")

That percentage may seem small, but at scale it adds up. If you are processing millions of documents through an LLM API at a cost of a few pence per thousand tokens, invisible characters represent a measurable and entirely avoidable expense.

## Step 4: PII redaction

Some of the digitised texts contain personal information that should not be sent to an external LLM API: email addresses, phone numbers, and similar. We need to detect and redact these.

Regular expressions are the right tool here. They are not perfect — no regex will catch every possible email format — but they handle the common cases well enough for a first pass.

In [ ]:
# Email pattern
email_pattern = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")

# UK phone number patterns
phone_pattern = re.compile(
    r"(?:\+44[\s-]?(?:\(?0\)?[\s-]?)?|0)\d{2,4}[\s-]?\d{3,4}[\s-]?\d{3,4}"
)

# Test on some examples
test_text = "Contact j.smith@britishlibrary.uk or call +44 20 7946 0958 for access."
print("Emails found:", email_pattern.findall(test_text))
print("Phones found:", phone_pattern.findall(test_text))

In [ ]:
def redact_pii(text):
    """Replace email addresses and phone numbers with redaction markers."""
    text = email_pattern.sub("[EMAIL REDACTED]", text)
    text = phone_pattern.sub("[PHONE REDACTED]", text)
    return text

# Test it
print(redact_pii(test_text))

In [ ]:
# How many texts contain PII?
has_email = df["raw_text"].apply(lambda t: bool(email_pattern.search(t)))
has_phone = df["raw_text"].apply(lambda t: bool(phone_pattern.search(t)))

print(f"Texts with emails: {has_email.sum()}")
print(f"Texts with phones: {has_phone.sum()}")
print(f"Texts with any PII: {(has_email | has_phone).sum()}")

## Step 5: Boilerplate removal

Many digitised pages carry repeated headers and footers — "Page X of Y — British Library Digital Collection" and similar. These are useful for navigation in the original digital viewer but are noise in a search index.

We can detect and remove common boilerplate patterns.

In [ ]:
# Common boilerplate patterns in the collection
boilerplate_patterns = [
    re.compile(r"Page \d+ of \d+ -- British Library Digital Collection"),
    re.compile(r"--- British Library Digitised Archive --- Page \d+"),
    re.compile(r"BL Digital Collections \| Catalogue Ref: [A-Z/0-9]+ \| Page \d+"),
    re.compile(r"\[Digitised by the British Library, \d{4}\]"),
]

def remove_boilerplate(text):
    """Remove known boilerplate headers and footers."""
    for pattern in boilerplate_patterns:
        text = pattern.sub("", text)
    # Clean up extra whitespace left behind
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Test it
test_boilerplate = "Page 42 of 200 -- British Library Digital Collection\n\nThe feudal system in England..."
print("Before:", repr(test_boilerplate[:80]))
print("After: ", repr(remove_boilerplate(test_boilerplate)[:80]))

In [ ]:
# Count texts with boilerplate
has_boilerplate = df["raw_text"].apply(
    lambda t: any(p.search(t) for p in boilerplate_patterns)
)
print(f"Texts with boilerplate: {has_boilerplate.sum()} ({has_boilerplate.mean():.1%})")

## Building the complete pipeline

We have five cleaning steps. Now we chain them into a single function that processes any raw text through the full pipeline. The order matters: we unescape HTML entities before stripping tags, and we normalise Unicode before looking for invisible characters.

In [ ]:
def clean_text(text):
    """Full text cleaning pipeline for British Library digitised texts.
    
    Steps:
    1. HTML entity unescaping and tag removal
    2. Unicode normalisation (NFKC)
    3. Invisible character removal
    4. PII redaction
    5. Boilerplate removal
    6. Whitespace normalisation
    """
    # 1. HTML
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", "", text)
    
    # 2. Unicode normalisation
    text = unicodedata.normalize("NFKC", text)
    
    # 3. Invisible characters
    for char in ["\xa0", "\u200b", "\u200c", "\u200d", "\ufeff"]:
        text = text.replace(char, " " if char == "\xa0" else "")
    
    # 4. PII redaction
    text = redact_pii(text)
    
    # 5. Boilerplate removal
    text = remove_boilerplate(text)
    
    # 6. Whitespace normalisation
    text = re.sub(r" {2,}", " ", text)   # collapse multiple spaces
    text = re.sub(r"\n{3,}", "\n\n", text)  # collapse excessive newlines
    text = text.strip()
    
    return text

In [ ]:
# Test the full pipeline on a messy example
messy = (
    "Page 7 of 45 -- British Library Digital Collection\n\n"
    "<p>The Black\xa0Death reached England\u200b in June 1348, "
    "arriving through the port of <em>Melcombe Regis</em> in Dorset.</p>\n"
    "Contact curator@bl.uk for access to the original manuscript.\n"
    "Call +44 20 7946 0958 for appointments."
)

print("=== RAW ===")
print(messy)
print()
print("=== CLEANED ===")
print(clean_text(messy))
print()
print("=== repr() RAW ===")
print(repr(messy[:100]))
print()
print("=== repr() CLEANED ===")
print(repr(clean_text(messy)[:100]))

## Applying the pipeline to the full dataset

Now we run every text through the cleaning pipeline and measure the results.

In [ ]:
# Apply the pipeline
df["clean_text"] = df["raw_text"].apply(clean_text)

# Compare a few before and after
for i in [0, 50, 100]:
    print(f"--- {df.iloc[i]['text_id']} ---")
    print(f"RAW:   {df.iloc[i]['raw_text'][:150]}...")
    print(f"CLEAN: {df.iloc[i]['clean_text'][:150]}...")
    print()

## Cleaning metrics

Good data engineering means measuring the impact of your transformations. How many characters did we remove? How many tokens did we save? What was the processing rate?

In [ ]:
import time

# Character counts
raw_chars = df["raw_text"].str.len().sum()
clean_chars = df["clean_text"].str.len().sum()

# Token counts (rough)
raw_tokens = df["raw_text"].apply(count_rough_tokens).sum()
clean_tokens = df["clean_text"].apply(count_rough_tokens).sum()

# Processing rate
start = time.time()
_ = df["raw_text"].apply(clean_text)
elapsed = time.time() - start
rate = len(df) / elapsed

print("=== Cleaning Metrics ===")
print(f"Characters: {raw_chars:,} -> {clean_chars:,} (removed {raw_chars - clean_chars:,}, {(raw_chars - clean_chars)/raw_chars:.1%})")
print(f"Tokens:     {raw_tokens:,} -> {clean_tokens:,} (saved {raw_tokens - clean_tokens:,}, {(raw_tokens - clean_tokens)/raw_tokens:.1%})")
print(f"Rate:       {rate:.0f} texts/second")

## A closer look at what changed

Let us categorise the cleaning impact per text.

In [ ]:
df["raw_len"] = df["raw_text"].str.len()
df["clean_len"] = df["clean_text"].str.len()
df["chars_removed"] = df["raw_len"] - df["clean_len"]
df["pct_removed"] = df["chars_removed"] / df["raw_len"]

print("Characters removed per text:")
print(df["chars_removed"].describe().to_string())
print()
print(f"Texts where >20% of characters were removed: {(df['pct_removed'] > 0.2).sum()}")
print(f"Texts where nothing changed: {(df['chars_removed'] == 0).sum()}")

---

## Exercises

### Exercise 1: Detect OCR artefacts

Some texts contain OCR errors where "rn" has been misread as "m", or "l" (lowercase L) has been misread as "1" (the digit one). Write a function called `count_ocr_suspects` that takes a text and returns the number of words containing a digit surrounded by letters (e.g., "1ife" instead of "life"). This is a rough heuristic, not a perfect detector.

Test it on the raw texts. How many texts contain at least one suspect word?

In [ ]:
# Your code here


### Exercise 2: Add a date-of-birth redaction step

Extend the `redact_pii` function to also detect and redact dates in the format DD/MM/YYYY (e.g., "15/03/1990"). Replace them with `[DATE REDACTED]`.

Be careful: you do not want to redact historical dates that are part of the content (e.g., years like "1348" on their own). Only target the full DD/MM/YYYY format.

In [ ]:
# Your code here


### Exercise 3: Measure the impact of each cleaning step individually

Rather than running the full pipeline at once, apply each step in isolation and measure how many characters it removes on average. Which single step has the biggest impact? Present your results in a pandas DataFrame with columns: `step`, `avg_chars_removed`, `texts_affected`.

In [ ]:
# Your code here


### Your turn: Design a new cleaning step

Browse through the raw texts and identify a pattern of noise that our current pipeline does not handle. It could be stray punctuation, repeated whitespace patterns, encoding artefacts, or anything else you spot.

1. Describe the pattern you found
2. Write a function to detect and clean it
3. Measure its impact across the dataset
4. Integrate it into the `clean_text` pipeline

In [ ]:
# Your code here


---

## Summary

In this notebook we built a text cleaning pipeline with five stages:

1. **HTML removal** — `html.unescape()` and regex tag stripping
2. **Unicode normalisation** — `unicodedata.normalize('NFKC', text)` to canonicalise character representations
3. **Invisible character removal** — `repr()` reveals `\xa0` and `\u200b` hiding in plain sight; removing them saves tokens and money
4. **PII redaction** — regex patterns for emails and phone numbers
5. **Boilerplate removal** — strip repeated headers and footers from digitised pages

The big idea: `repr()` is your friend. Text that looks clean to the naked eye may be full of invisible characters that inflate token counts and waste money. Always inspect with `repr()` before trusting a text.

Next up: once texts are clean, we need to deal with duplicates. The same manuscript scanned three times under different catalogue numbers is the same document, and our search system should not return it three times.